# Thesis 08: Across-Experiment Comparison

Collect the thesis result artifacts into one analysis surface so differences between dataset generation, optimizer, loss balancing, collocation, multistage, data augmentation, and final experiments can be compared consistently. This notebook starts from the exported `results/tables/*` tables created by the preceding analysis notebooks.

## Imports

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "src").is_dir():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            return candidate
    raise FileNotFoundError(f"Could not find repository root from {cwd}")


REPO = find_repo_root()

from results.notebook_utils import table_dir, write_table  # noqa: E402
from src.visualization.thesis_style import (  # noqa: E402
    DTU_COLORS,
    NEUTRAL_COLORS,
    THESIS_COLOR_CYCLE,
    apply_thesis_axis_style,
    log_minor_grid,
    set_thesis_style,
)

set_thesis_style()

TABLES = REPO / "results" / "tables"
TABLE_EXPORT = table_dir("across_experiments")

## Load Exported Thesis Tables

The earlier notebooks do the experiment-specific parsing and posthoc overlay. Here we keep those semantics and only harmonize common fields across experiments.

In [ ]:
SUMMARY_SPECS = [
    {
        "experiment_id": "01_dataset_generation",
        "experiment_label": "Dataset generation",
        "path": TABLES / "01_dataset_generation" / "main_numeric.csv",
        "label_cols": ["budget", "method"],
        "metric_prefix": "direct",
    },
    {
        "experiment_id": "02_optimizer_comparison",
        "experiment_label": "Optimizer comparison",
        "path": TABLES / "02_optimizer_comparison" / "main_numeric.csv",
        "label_cols": ["analysis_key"],
        "metric_prefix": "best",
    },
    {
        "experiment_id": "03_loss_balancing",
        "experiment_label": "Loss balancing",
        "path": TABLES / "03_loss_balancing" / "main_numeric.csv",
        "label_cols": ["strategy"],
        "metric_prefix": "best",
    },
    {
        "experiment_id": "04_collocation_comparison",
        "experiment_label": "Collocation sampling",
        "path": TABLES / "04_collocation_comparison" / "main_numeric.csv",
        "label_cols": ["density_label", "strategy"],
        "metric_prefix": "best",
    },
    {
        "experiment_id": "05_multistage",
        "experiment_label": "Multistage training",
        "path": TABLES / "05_multistage" / "main_numeric.csv",
        "label_cols": ["strategy"],
        "metric_prefix": "best",
    },
    {
        "experiment_id": "06_data_augmentation",
        "experiment_label": "Data augmentation",
        "path": TABLES / "06_data_augmentation" / "main_numeric.csv",
        "label_cols": ["supervised_strategy", "collocation_strategy"],
        "metric_prefix": "best",
    },
    {
        "experiment_id": "07_final_experiment",
        "experiment_label": "Final experiment",
        "path": TABLES / "07_final_experiment" / "full_external_numeric.csv",
        "label_cols": ["model_label", "strategy_label"],
        "metric_prefix": "best",
        "filter_col": "model_flag",
        "filter_value": "SM4",
    },
]


def read_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)


def make_label(frame: pd.DataFrame, columns: list[str]) -> pd.Series:
    existing = [col for col in columns if col in frame.columns]
    if not existing:
        return pd.Series(np.arange(len(frame)).astype(str), index=frame.index)
    return frame[existing].astype(str).agg(" / ".join, axis=1)


summary_frames = []
inventory_rows = []
for spec in SUMMARY_SPECS:
    frame = read_csv(spec["path"])
    inventory_rows.append({
        "artifact_type": "summary",
        "experiment_id": spec["experiment_id"],
        "experiment_label": spec["experiment_label"],
        "path": str(spec["path"].relative_to(REPO)),
        "exists": spec["path"].exists(),
        "rows": len(frame),
        "columns": frame.shape[1],
        "filter": "" if "filter_col" not in spec else f"{spec['filter_col']} == {spec['filter_value']}",
    })
    if frame.empty:
        continue
    frame = frame.copy()
    if spec.get("filter_col") in frame.columns:
        frame = frame[frame[spec["filter_col"]].astype(str) == str(spec["filter_value"])].copy()
    frame.insert(0, "experiment_id", spec["experiment_id"])
    frame.insert(1, "experiment_label", spec["experiment_label"])
    frame.insert(2, "configuration", make_label(frame, spec["label_cols"]))
    frame.insert(3, "metric_prefix", spec["metric_prefix"])
    summary_frames.append(frame)

summary_raw = pd.concat(summary_frames, ignore_index=True, sort=False) if summary_frames else pd.DataFrame()
artifact_inventory = pd.DataFrame(inventory_rows)

display(artifact_inventory)

## Harmonize Common Metrics

In [ ]:
COMMON_METRICS = {
    "id_mse": ["id_mse_mean", "best_id_eval_mse_mean"],
    "id_mse_sem": ["id_mse_sem", "best_id_eval_mse_sem"],
    "ood_mse": ["ood_mse_mean", "best_ood_eval_mse_mean"],
    "ood_mse_sem": ["ood_mse_sem", "best_ood_eval_mse_sem"],
    "id_mae": ["id_mae_mean", "best_id_eval_mae_mean"],
    "ood_mae": ["ood_mae_mean", "best_ood_eval_mae_mean"],
    "id_max_abs_error": ["id_max_abs_error_mean", "best_id_eval_max_abs_error_mean"],
    "ood_max_abs_error": ["ood_max_abs_error_mean", "best_ood_eval_max_abs_error_mean"],
    "id_trajectory_mse_p95": ["id_trajectory_mse_p95_mean", "best_id_eval_trajectory_mse_p95_mean"],
    "ood_trajectory_mse_p95": ["ood_trajectory_mse_p95_mean", "best_ood_eval_trajectory_mse_p95_mean"],
    "id_trajectory_mse_max": ["id_trajectory_mse_max_mean", "best_id_eval_trajectory_mse_max_mean"],
    "ood_trajectory_mse_max": ["ood_trajectory_mse_max_mean", "best_ood_eval_trajectory_mse_max_mean"],
    "training_seconds": ["training_seconds_mean"],
    "total_seconds": ["total_seconds_mean"],
    "n_runs": ["n_runs"],
}


def first_available(row: pd.Series, candidates: list[str]) -> float:
    for col in candidates:
        if col in row.index and pd.notna(row[col]):
            return row[col]
    return np.nan


records = []
for _, row in summary_raw.iterrows():
    out = {
        "experiment_id": row["experiment_id"],
        "experiment_label": row["experiment_label"],
        "configuration": row["configuration"],
        "metric_prefix": row["metric_prefix"],
    }
    for metric, candidates in COMMON_METRICS.items():
        out[metric] = first_available(row, candidates)
    if pd.isna(out["training_seconds"]):
        hours = first_available(row, ["training_hours_mean"])
        out["training_seconds"] = np.nan if pd.isna(hours) else float(hours) * 3600.0
    out["ood_id_mse_gap"] = out["ood_mse"] - out["id_mse"]
    out["ood_id_mse_ratio"] = out["ood_mse"] / out["id_mse"] if pd.notna(out["id_mse"]) and out["id_mse"] != 0 else np.nan
    records.append(out)

summary_metrics = pd.DataFrame(records)
# Scratch export disabled for repository submission; keep derived data in memory.

metric_availability = (
    summary_metrics.groupby("experiment_label", observed=True)
    .agg(**{metric: (metric, lambda s: int(s.notna().sum())) for metric in COMMON_METRICS})
    .reset_index()
)
# Scratch export disabled for repository submission; keep derived data in memory.

display(summary_metrics.head(12))
display(metric_availability)

## Best Configuration Per Experiment

This first pass ranks configurations by ID MSE and OOD MSE separately. The comparison is diagnostic: experiments do not all use identical model scopes or design spaces.

In [ ]:
def best_rows(frame: pd.DataFrame, metric: str) -> pd.DataFrame:
    rows = []
    for experiment_label, sub in frame.dropna(subset=[metric]).groupby("experiment_label", observed=True):
        best = sub.sort_values(metric, ascending=True).iloc[0].copy()
        best["ranking_metric"] = metric
        rows.append(best)
    return pd.DataFrame(rows)


best_by_experiment = pd.concat(
    [best_rows(summary_metrics, "id_mse"), best_rows(summary_metrics, "ood_mse")],
    ignore_index=True,
    sort=False,
)
best_by_experiment = best_by_experiment[
    [
        "ranking_metric", "experiment_id", "experiment_label", "configuration",
        "id_mse", "ood_mse", "ood_id_mse_gap", "ood_id_mse_ratio",
        "id_trajectory_mse_p95", "ood_trajectory_mse_p95", "training_seconds", "n_runs",
    ]
]
# Scratch export disabled for repository submission; keep derived data in memory.
display(best_by_experiment)

## Starter Cross-Experiment Views

In [ ]:
plot_data = best_by_experiment[best_by_experiment["ranking_metric"] == "id_mse"].copy()
plot_data = plot_data.sort_values("id_mse", ascending=False)

fig, ax = plt.subplots(figsize=(9.0, 4.6))
ax.barh(plot_data["experiment_label"], plot_data["id_mse"], color="#4C78A8")
ax.set_xscale("log")
ax.set_xlabel("best ID MSE within experiment")
ax.set_ylabel("")
ax.grid(True, axis="x", alpha=0.35)
fig.tight_layout()
# Figure export disabled for repository submission; keep plot inline.
plt.show()

tradeoff = summary_metrics.dropna(subset=["id_mse", "ood_mse"]).copy()
fig, ax = plt.subplots(figsize=(6.6, 5.2))
for experiment_label, sub in tradeoff.groupby("experiment_label", observed=True):
    ax.scatter(sub["id_mse"], sub["ood_mse"], label=experiment_label, alpha=0.78, s=42)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("ID MSE")
ax.set_ylabel("OOD MSE")
ax.grid(True, which="both", alpha=0.3)
ax.legend(fontsize=8, loc="best")
fig.tight_layout()
# Figure export disabled for repository submission; keep plot inline.
plt.show()

## Best-Result Metric Comparisons

For each main metric, select the best SM4 configuration within each experiment by the corresponding ID metric, then compare its ID and OOD performance. The final experiment is restricted to `model_flag == "SM4"` so it is comparable with the SM4-focused earlier experiments.


In [ ]:
MAIN_METRIC_PAIRS = [
    ("mse", "id_mse", "ood_mse", "MSE"),
    ("mae", "id_mae", "ood_mae", "MAE"),
    ("max_abs_error", "id_max_abs_error", "ood_max_abs_error", "max absolute error"),
    ("trajectory_mse_p95", "id_trajectory_mse_p95", "ood_trajectory_mse_p95", "trajectory MSE p95"),
    ("trajectory_mse_max", "id_trajectory_mse_max", "ood_trajectory_mse_max", "trajectory MSE max"),
]

best_metric_rows = []
for metric_key, id_col, ood_col, label in MAIN_METRIC_PAIRS:
    usable = summary_metrics.dropna(subset=[id_col, ood_col]).copy()
    for experiment_label, sub in usable.groupby("experiment_label", observed=True):
        best = sub.sort_values(id_col, ascending=True).iloc[0].copy()
        best_metric_rows.append({
            "metric": metric_key,
            "metric_label": label,
            "experiment_id": best["experiment_id"],
            "experiment_label": experiment_label,
            "configuration": best["configuration"],
            "id_value": best[id_col],
            "ood_value": best[ood_col],
            "ood_id_gap": best[ood_col] - best[id_col],
            "ood_id_ratio": best[ood_col] / best[id_col] if best[id_col] != 0 else np.nan,
            "training_seconds": best.get("training_seconds", np.nan),
            "n_runs": best.get("n_runs", np.nan),
        })

best_metric_comparisons = pd.DataFrame(best_metric_rows)
# Scratch export disabled for repository submission; keep derived data in memory.
display(best_metric_comparisons)


def sci(value: float, digits: int = 2) -> str:
    if pd.isna(value):
        return ""
    return f"{float(value):.{digits}e}"


def ratio_text(value: float) -> str:
    if pd.isna(value):
        return ""
    return f"{float(value):.1f}x"

best_metric_summary_table = pd.DataFrame({
    "metric": best_metric_comparisons["metric_label"],
    "experiment": best_metric_comparisons["experiment_label"],
    "best configuration": best_metric_comparisons["configuration"],
    "ID value": best_metric_comparisons["id_value"].map(sci),
    "OOD value": best_metric_comparisons["ood_value"].map(sci),
    "OOD / ID": best_metric_comparisons["ood_id_ratio"].map(ratio_text),
    "training time [h]": (best_metric_comparisons["training_seconds"] / 3600.0).map(
        lambda value: "not available" if pd.isna(value) else f"{float(value):.2f}"
    ),
    "runs": best_metric_comparisons["n_runs"].map(lambda value: "not available" if pd.isna(value) else f"{int(value)}"),
})
write_table(best_metric_summary_table, "across_experiments")
display(best_metric_summary_table)

EXPERIMENT_ORDER = [
    "Dataset generation",
    "Optimizer comparison",
    "Loss balancing",
    "Collocation sampling",
    "Multistage training",
    "Data augmentation",
    "Final experiment",
]
EXPERIMENT_COLORS = {
    label: THESIS_COLOR_CYCLE[idx % len(THESIS_COLOR_CYCLE)]
    for idx, label in enumerate(EXPERIMENT_ORDER)
}
EXPERIMENT_MARKERS = {
    "Dataset generation": "o",
    "Optimizer comparison": "s",
    "Loss balancing": "^",
    "Collocation sampling": "D",
    "Multistage training": "P",
    "Data augmentation": "X",
    "Final experiment": "*",
}

n_metrics = len(MAIN_METRIC_PAIRS)
fig, axes = plt.subplots(
    2,
    3,
    figsize=(7.35, 4.85),
    constrained_layout=True,
)
axes = axes.ravel()
for ax, (metric_key, _id_col, _ood_col, label) in zip(axes, MAIN_METRIC_PAIRS):
    sub = best_metric_comparisons[best_metric_comparisons["metric"] == metric_key].copy()
    sub["experiment_label"] = pd.Categorical(sub["experiment_label"], categories=EXPERIMENT_ORDER, ordered=True)
    sub = sub.sort_values("experiment_label")
    for _, row in sub.iterrows():
        experiment = str(row["experiment_label"])
        ax.scatter(
            row["id_value"],
            row["ood_value"],
            s=42 if experiment != "Final experiment" else 58,
            marker=EXPERIMENT_MARKERS.get(experiment, "o"),
            color=EXPERIMENT_COLORS.get(experiment, NEUTRAL_COLORS["muted"]),
            edgecolor=NEUTRAL_COLORS["axis"],
            linewidth=0.45,
            alpha=0.92,
            zorder=3,
        )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(f"ID {label}")
    ax.set_ylabel(f"OOD {label}")
    ax.set_title(label)
    ax.grid(True, which="major", alpha=0.65)
    log_minor_grid(ax, axis="both", alpha=0.45, linewidth=0.25)
    apply_thesis_axis_style(ax)

legend_ax = axes[n_metrics]
legend_ax.axis("off")
legend_handles = [
    plt.Line2D(
        [0],
        [0],
        marker=EXPERIMENT_MARKERS.get(label, "o"),
        linestyle="none",
        markerfacecolor=EXPERIMENT_COLORS[label],
        markeredgecolor=NEUTRAL_COLORS["axis"],
        markeredgewidth=0.45,
        markersize=6.2 if label != "Final experiment" else 7.4,
        label=label,
    )
    for label in EXPERIMENT_ORDER
]
legend_ax.legend(
    handles=legend_handles,
    loc="center left",
    title="Experiment",
    frameon=False,
    borderaxespad=0.0,
    labelspacing=0.55,
)

for ax in axes[n_metrics + 1:]:
    ax.axis("off")

# Figure export disabled for repository submission; keep plot inline.
plt.show()

# Same publication figure without the final experiment, which otherwise dominates
# the axis scale and visually compresses the method-development experiments.
comparison_without_final = best_metric_comparisons[
    best_metric_comparisons["experiment_label"] != "Final experiment"
].copy()
experiment_order_without_final = [label for label in EXPERIMENT_ORDER if label != "Final experiment"]

fig, axes = plt.subplots(
    2,
    3,
    figsize=(7.35, 4.85),
    constrained_layout=True,
)
axes = axes.ravel()
for ax, (metric_key, _id_col, _ood_col, label) in zip(axes, MAIN_METRIC_PAIRS):
    sub = comparison_without_final[comparison_without_final["metric"] == metric_key].copy()
    sub["experiment_label"] = pd.Categorical(
        sub["experiment_label"], categories=experiment_order_without_final, ordered=True
    )
    sub = sub.sort_values("experiment_label")
    for _, row in sub.iterrows():
        experiment = str(row["experiment_label"])
        ax.scatter(
            row["id_value"],
            row["ood_value"],
            s=42,
            marker=EXPERIMENT_MARKERS.get(experiment, "o"),
            color=EXPERIMENT_COLORS.get(experiment, NEUTRAL_COLORS["muted"]),
            edgecolor=NEUTRAL_COLORS["axis"],
            linewidth=0.45,
            alpha=0.92,
            zorder=3,
        )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(f"ID {label}")
    ax.set_ylabel(f"OOD {label}")
    ax.set_title(label)
    ax.grid(True, which="major", alpha=0.65)
    log_minor_grid(ax, axis="both", alpha=0.45, linewidth=0.25)
    apply_thesis_axis_style(ax)

legend_ax = axes[n_metrics]
legend_ax.axis("off")
legend_handles = [
    plt.Line2D(
        [0],
        [0],
        marker=EXPERIMENT_MARKERS.get(label, "o"),
        linestyle="none",
        markerfacecolor=EXPERIMENT_COLORS[label],
        markeredgecolor=NEUTRAL_COLORS["axis"],
        markeredgewidth=0.45,
        markersize=6.2,
        label=label,
    )
    for label in experiment_order_without_final
]
legend_ax.legend(
    handles=legend_handles,
    loc="center left",
    title="Experiment",
    frameon=False,
    borderaxespad=0.0,
    labelspacing=0.55,
)

for ax in axes[n_metrics + 1:]:
    ax.axis("off")

# Figure export disabled for repository submission; keep plot inline.
plt.show()

# Compact thesis figure with the two headline metrics and a dedicated legend column.
HEADLINE_METRIC_PAIRS = [
    ("mse", "id_mse", "ood_mse", "MSE"),
    ("max_abs_error", "id_max_abs_error", "ood_max_abs_error", "Max absolute error"),
]


def plot_headline_metric_pair(
    frame: pd.DataFrame,
    *,
    experiment_order: list[str],
    filename: str,
) -> None:
    fig, axes = plt.subplots(
        1,
        3,
        figsize=(7.35, 2.55),
        constrained_layout=True,
        gridspec_kw={"width_ratios": [1.0, 1.0, 0.86]},
    )
    for col_idx, (ax, (metric_key, _id_col, _ood_col, title)) in enumerate(zip(axes[:2], HEADLINE_METRIC_PAIRS)):
        sub = frame[frame["metric"] == metric_key].copy()
        sub["experiment_label"] = pd.Categorical(
            sub["experiment_label"], categories=experiment_order, ordered=True
        )
        sub = sub.sort_values("experiment_label")
        for _, row in sub.iterrows():
            experiment = str(row["experiment_label"])
            ax.scatter(
                row["id_value"],
                row["ood_value"],
                s=42 if experiment != "Final experiment" else 58,
                marker=EXPERIMENT_MARKERS.get(experiment, "o"),
                color=EXPERIMENT_COLORS.get(experiment, NEUTRAL_COLORS["muted"]),
                edgecolor=NEUTRAL_COLORS["axis"],
                linewidth=0.45,
                alpha=0.92,
                zorder=3,
            )
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("ID")
        ax.set_ylabel("OOD" if col_idx == 0 else "")
        ax.set_title(title)
        ax.grid(True, which="major", alpha=0.65)
        log_minor_grid(ax, axis="both", alpha=0.45, linewidth=0.25)
        apply_thesis_axis_style(ax)

    legend_ax = axes[2]
    legend_ax.axis("off")
    legend_handles = [
        plt.Line2D(
            [0],
            [0],
            marker=EXPERIMENT_MARKERS.get(label, "o"),
            linestyle="none",
            markerfacecolor=EXPERIMENT_COLORS[label],
            markeredgecolor=NEUTRAL_COLORS["axis"],
            markeredgewidth=0.45,
            markersize=6.2 if label != "Final experiment" else 7.4,
            label=label,
        )
        for label in experiment_order
    ]
    legend_ax.legend(
        handles=legend_handles,
        loc="center left",
        title="Experiment",
        frameon=False,
        borderaxespad=0.0,
        labelspacing=0.55,
    )
    # Figure export disabled for repository submission; keep plot inline.
    plt.show()


plot_headline_metric_pair(
    best_metric_comparisons,
    experiment_order=EXPERIMENT_ORDER,
    filename="headline_mse_max_abs_id_vs_ood",
)
plot_headline_metric_pair(
    comparison_without_final,
    experiment_order=experiment_order_without_final,
    filename="headline_mse_max_abs_id_vs_ood_without_final",
)

for metric_key, _id_col, _ood_col, label in MAIN_METRIC_PAIRS:
    sub = best_metric_comparisons[best_metric_comparisons["metric"] == metric_key].copy()
    sub = sub.sort_values("id_value", ascending=False)
    fig, ax = plt.subplots(figsize=(7.8, 4.6))
    ax.scatter(sub["id_value"], sub["ood_value"], s=58, alpha=0.86, color="#4C78A8")
    for _, row in sub.iterrows():
        ax.annotate(
            row["experiment_label"],
            (row["id_value"], row["ood_value"]),
            xytext=(5, 3),
            textcoords="offset points",
            fontsize=8,
        )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(f"ID {label}")
    ax.set_ylabel(f"OOD {label}")
    ax.grid(True, which="both", alpha=0.3)
    ax.set_title(f"Best experiment result by ID {label}")
    fig.tight_layout()
    # Figure export disabled for repository submission; keep plot inline.
    plt.show()